# Introduction to Supervised Learning

**What this notebook covers:**
- What supervised learning is and how it works
- Regression vs Classification tasks
- Building a simple classifier from scratch using NumPy
- Comparing it with scikit-learn's implementation
- Visualizing decision boundaries and performance

**Prerequisites:**
- Basic Python (functions, classes, loops)
- NumPy and Pandas basics
- Basic statistics (mean, variance)

---

**Dataset:** Titanic - Machine Learning from Disaster  
🔗 [Kaggle Dataset Link](https://www.kaggle.com/competitions/titanic/data)  
*Credits: Kaggle / Titanic competition dataset — a classic supervised learning dataset with real passenger survival records.*

In [ ]:
# --- All Imports ---
import numpy as np                        # Numerical operations and array math
import pandas as pd                       # Data loading and manipulation
import matplotlib.pyplot as plt           # Plotting and visualization
import seaborn as sns                     # Statistical visualizations
from sklearn.linear_model import LogisticRegression   # Sklearn classifier
from sklearn.model_selection import train_test_split, cross_val_score  # Data splitting and CV
from sklearn.preprocessing import StandardScaler      # Feature scaling
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Evaluation

np.random.seed(42)  # For reproducibility across all random operations

## Part 1: Theory Recap

Before we touch any data, here is a 5-bullet summary of what supervised learning is:

- **Supervised learning** trains a model on labeled data — input-output pairs — so it can predict outputs for new, unseen inputs.
- The model learns a function **f: X → y** by minimizing a **loss function** that measures prediction error.
- **Regression** predicts continuous values (e.g., house price); **Classification** predicts discrete categories (e.g., survived/not survived).
- The **bias-variance tradeoff** is central: a too-simple model underfits; a too-complex model overfits. We always evaluate on a held-out test set.
- Common evaluation metrics: **MSE/R²** for regression, **Accuracy/F1/AUC-ROC** for classification.

## Loading the Dataset

We use the **Titanic dataset** — a perfect introduction to supervised classification.
- **Input features (X):** Passenger age, class, fare, sex, etc.
- **Target variable (y):** `Survived` — 1 if the passenger survived, 0 if not.

This is a binary classification problem: given passenger info, predict survival.

In [ ]:
# Load dataset directly from a public URL (same data as Kaggle Titanic)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print("Shape:", df.shape)
print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Dataset Info ---")
df.info()

print("\n--- Statistical Summary ---")
display(df.describe())

print("\n--- Target Variable Distribution ---")
print(df['Survived'].value_counts())
print("Survival Rate:", df['Survived'].mean().round(3))

## Preprocessing

Real-world data is messy. We need to handle:
1. **Missing values** — Age, Cabin, Embarked have nulls
2. **Categorical variables** — Sex, Embarked need to be encoded as numbers
3. **Feature selection** — Drop irrelevant columns (Name, Ticket, Cabin)
4. **Feature scaling** — Normalize numerical features so no one feature dominates

In [ ]:
# Step 1: Drop columns that are not useful for prediction
df = df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

# Step 2: Fill missing Age with median (robust to outliers)
df['Age'].fillna(df['Age'].median(), inplace=True)

# Step 3: Fill missing Embarked with mode (most common port)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Step 4: Encode categorical variables as integers
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})       # Binary encoding
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})  # Ordinal encoding

# Step 5: Separate features and target
X = df.drop('Survived', axis=1).values   # Feature matrix
y = df['Survived'].values                 # Target vector

# Step 6: Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 7: Scale features (mean=0, std=1) — important for distance-based and gradient methods
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit on train, transform train
X_test = scaler.transform(X_test)        # Only transform test (no data leakage!)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("Missing values remaining:", df.isnull().sum().sum())

## Part 2: From Scratch Implementation

We will implement a **K-Nearest Neighbors (KNN) classifier** from scratch using only NumPy.

KNN is the perfect algorithm to introduce supervised learning because:
- It is extremely intuitive: "You are who your neighbors are"
- No complex math — just distance calculation and voting
- It perfectly demonstrates the core supervised learning loop: store training data → compare new points → predict

**How KNN works:**
1. Store all training examples
2. For a new point, calculate distance to all training points
3. Find the k closest neighbors
4. Return the majority class among those neighbors

In [ ]:
class KNNFromScratch:
    """
    K-Nearest Neighbors Classifier built from scratch using NumPy.
    Demonstrates the core supervised learning predict loop.
    """

    def __init__(self, k=5):
        # INTERVIEW NOTE: k is the key hyperparameter — controls the bias-variance tradeoff
        # Small k = low bias, high variance (overfitting)
        # Large k = high bias, low variance (underfitting)
        self.k = k

    def fit(self, X, y):
        # INTERVIEW NOTE: KNN is a lazy learner — it does NO computation during training.
        # It simply memorizes the training data. All work happens at prediction time.
        self.X_train = X
        self.y_train = y

    def _euclidean_distance(self, x1, x2):
        # Euclidean distance: sqrt of sum of squared differences
        # INTERVIEW NOTE: Distance metric choice affects KNN performance significantly
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict(self, X):
        # Predict class for each test point
        predictions = [self._predict_single(x) for x in X]
        return np.array(predictions)

    def _predict_single(self, x):
        # Step 1: Compute distance from x to every training point
        distances = [self._euclidean_distance(x, x_train) for x_train in self.X_train]

        # Step 2: Sort by distance and get indices of k nearest neighbors
        k_indices = np.argsort(distances)[:self.k]

        # Step 3: Get labels of those k neighbors
        k_labels = self.y_train[k_indices]

        # Step 4: Majority vote — return the most common class
        # INTERVIEW NOTE: This is classification by plurality voting
        return np.bincount(k_labels).argmax()

# Instantiate and train
knn_scratch = KNNFromScratch(k=5)
knn_scratch.fit(X_train, y_train)

print("KNN from scratch trained successfully!")
print(f"Training examples stored: {len(knn_scratch.X_train)}")
print(f"k = {knn_scratch.k}")

In [ ]:
# Evaluate the from-scratch KNN model
y_pred_scratch = knn_scratch.predict(X_test)

scratch_accuracy = accuracy_score(y_test, y_pred_scratch)
print("=== From Scratch KNN Results ===")
print(f"Accuracy: {scratch_accuracy:.4f} ({scratch_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_scratch, target_names=['Did Not Survive', 'Survived']))

## Part 3: Sklearn Implementation

Scikit-learn provides an optimized KNN implementation that uses KD-Trees and Ball Trees internally for faster distance computation — instead of the brute-force O(n) search we wrote above.

The API is identical: `fit()` → `predict()` → `score()`. The difference is speed and built-in optimization.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Sklearn KNN — same k=5 for fair comparison
knn_sklearn = KNeighborsClassifier(n_neighbors=5)
knn_sklearn.fit(X_train, y_train)
y_pred_sklearn = knn_sklearn.predict(X_test)

sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)

print("=== Sklearn KNN Results ===")
print(f"Accuracy: {sklearn_accuracy:.4f} ({sklearn_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_sklearn, target_names=['Did Not Survive', 'Survived']))

print("\n=== Comparison ===")
print(f"From Scratch Accuracy : {scratch_accuracy:.4f}")
print(f"Sklearn Accuracy      : {sklearn_accuracy:.4f}")
print(f"Difference            : {abs(scratch_accuracy - sklearn_accuracy):.4f}")
# Small differences may exist due to tie-breaking rules in distance computation

In [ ]:
# --- Visualisation 1: Confusion Matrix ---
# Shows where the model makes correct vs incorrect predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_scratch, y_pred_sklearn],
                              ['From Scratch KNN', 'Sklearn KNN']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Survived', 'Survived'],
                yticklabels=['Not Survived', 'Survived'])
    ax.set_title(f'Confusion Matrix — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

plt.suptitle('Supervised Learning: Model Evaluation on Titanic Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Visualisation 2: Survival Rate by Feature ---
# Understand which features matter most for the supervised task
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Survival by Sex
df_original = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
survival_by_sex = df_original.groupby('Sex')['Survived'].mean()
axes[0].bar(survival_by_sex.index, survival_by_sex.values, color=['steelblue', 'salmon'], edgecolor='black')
axes[0].set_title('Survival Rate by Gender', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gender', fontsize=11)
axes[0].set_ylabel('Survival Rate', fontsize=11)
axes[0].set_ylim(0, 1)
for i, v in enumerate(survival_by_sex.values):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)

# Survival by Passenger Class
survival_by_class = df_original.groupby('Pclass')['Survived'].mean()
axes[1].bar(['1st Class', '2nd Class', '3rd Class'], survival_by_class.values,
            color=['gold', 'silver', '#cd7f32'], edgecolor='black')
axes[1].set_title('Survival Rate by Passenger Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=11)
axes[1].set_ylabel('Survival Rate', fontsize=11)
axes[1].set_ylim(0, 1)
for i, v in enumerate(survival_by_class.values):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)

plt.suptitle('Feature Analysis: What Drove Survival?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

The most important hyperparameter in KNN is **k** (number of neighbors).

- **Small k (e.g., k=1):** Very sensitive to individual training points → high variance → overfitting
- **Large k (e.g., k=50):** Averages over many neighbors → high bias → underfitting
- **Sweet spot:** A k value that balances both — typically found through cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

# Experiment 1: Vary k from 1 to 30
k_values = range(1, 31)
train_accuracies = []
cv_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_acc = knn.score(X_train, y_train)
    cv_acc = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy').mean()
    train_accuracies.append(train_acc)
    cv_accuracies.append(cv_acc)

# Experiment 2: Vary distance metric
metrics = ['euclidean', 'manhattan', 'chebyshev']
metric_scores = []
for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    score = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy').mean()
    metric_scores.append(score)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: k vs Accuracy
axes[0].plot(k_values, train_accuracies, label='Train Accuracy', color='blue', marker='o', markersize=4)
axes[0].plot(k_values, cv_accuracies, label='CV Accuracy (5-fold)', color='orange', marker='s', markersize=4)
best_k = k_values[np.argmax(cv_accuracies)]
axes[0].axvline(x=best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
axes[0].set_title('Effect of k on Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Neighbors (k)', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Distance Metric Comparison
bars = axes[1].bar(metrics, metric_scores, color=['steelblue', 'salmon', 'green'], edgecolor='black')
axes[1].set_title('Effect of Distance Metric on CV Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Distance Metric', fontsize=11)
axes[1].set_ylabel('5-Fold CV Accuracy', fontsize=11)
axes[1].set_ylim(0.6, 0.85)
for bar, score in zip(bars, metric_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{score:.3f}', ha='center', fontsize=11)

plt.suptitle('Hyperparameter Experiments — KNN on Titanic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best k found: {best_k} with CV accuracy: {max(cv_accuracies):.4f}")
print(f"Best distance metric: {metrics[np.argmax(metric_scores)]} with CV accuracy: {max(metric_scores):.4f}")

## Part 5: Interview Corner

**Question: What is supervised learning and how does it fundamentally differ from unsupervised learning?**

This is one of the most common ML interview openers. Here is how to answer it at a FAANG level:

**Definition:**  
Supervised learning is a paradigm where a model learns a mapping from inputs X to outputs y using labeled training data. The "supervision" refers to the fact that the true outputs (labels) are provided during training, guiding the model to minimize a loss function.

**Key distinction from unsupervised:**  
- Supervised: labels exist → task is prediction (regression/classification)
- Unsupervised: no labels → task is structure discovery (clustering/dimensionality reduction)

**Follow-up: What are the core assumptions?**
- Training and test data come from the same distribution (i.i.d.)
- Labels are accurate
- Features are informative with respect to the target

**Common pitfall to mention:**  
Data leakage — accidentally using test data information during training leads to overly optimistic evaluation. Always fit preprocessors (scalers, encoders) only on training data.

## Key Takeaways

Remember these 5 bullets for placement interviews:

- **Supervised learning = labeled data + loss minimization**: The model learns f(X) → y by minimizing prediction error on known input-output pairs.
- **Regression vs Classification**: Continuous output → Regression (MSE, R²); Discrete output → Classification (Accuracy, F1, AUC-ROC).
- **Bias-Variance Tradeoff**: Simple models underfit (high bias); complex models overfit (high variance). Use cross-validation to find the sweet spot.
- **Never evaluate on training data**: Always use a held-out test set or cross-validation to get an honest estimate of real-world performance.
- **Data quality > model complexity**: Garbage in, garbage out. Clean, representative, correctly labeled data matters more than which algorithm you pick.